# 第 6 天 · 流式输出与多轮对话

> 📓 **这是第 6 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/6)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 6 天 · 不需要 Key：用一个"假模型"理解流式输出和多轮记忆"""
import asyncio


async def fake_llm(messages: list[dict]):
    """假模型：async 生成器，一次吐一小段，模拟 stream=True 的 chunk"""
    users = [m["content"] for m in messages if m["role"] == "user"]
    reply = f"我一共收到 {len(messages)} 条消息，你说过的话：{'、'.join(users)}"
    for i in range(0, len(reply), 3):          # 每次吐 3 个字，像一个 chunk
        await asyncio.sleep(0.05)              # 模拟网络间隔
        yield reply[i:i + 3]


async def chat_once(messages: list[dict], question: str) -> None:
    messages.append({"role": "user", "content": question})
    print(f"你：{question}\n助手：", end="", flush=True)
    parts: list[str] = []
    async for chunk in fake_llm(messages):     # ≈ Flux.subscribe / 逐行读 SSE
        print(chunk, end="", flush=True)       # 边收边打印
        parts.append(chunk)                    # 同时拼起来
    print("\n")
    messages.append({"role": "assistant", "content": "".join(parts)})  # 回答也要存回历史


async def main() -> None:
    messages = [{"role": "system", "content": "你是一个助手"}]
    for q in ["我叫 Jiawei", "我是 Java 程序员", "我叫什么？"]:
        await chat_once(messages, q)
    print("最终 messages 条数：", len(messages))

    print("\n---- 对照：不保存历史，每次只发当前这一句 ----")
    for q in ["我叫 Jiawei", "我叫什么？"]:
        await chat_once([{"role": "system", "content": "你是一个助手"}], q)


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = ["openai>=1.40", "python-dotenv>=1.0"]
# ///
"""第 6 天：真模型 + 流式输出 + 多轮记忆（脚本自动问 3 轮，网页和本地都能跑）"""
import asyncio
import os
import sys
import time

from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, AsyncOpenAI, AuthenticationError

load_dotenv()
API_KEY = os.getenv("LLM_API_KEY")
BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com")
MODEL = os.getenv("LLM_MODEL", "deepseek-chat")

SYSTEM = "你是一位耐心的 Agent 开发老师，学生是 Java 后端程序员。每次回答不超过 80 字。"


async def ask_stream(client: AsyncOpenAI, messages: list[dict]) -> str:
    """发出整段历史，流式打印回答，返回完整回答文本"""
    start = time.perf_counter()
    first_ms = None
    parts: list[str] = []
    stream = await client.chat.completions.create(
        model=MODEL, messages=messages, stream=True, temperature=0.7, max_tokens=300
    )
    async for chunk in stream:                      # 每个 chunk 是一小段增量
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content or ""  # 注意：可能是 None
        if delta and first_ms is None:
            first_ms = (time.perf_counter() - start) * 1000
        print(delta, end="", flush=True)
        parts.append(delta)
    total_ms = (time.perf_counter() - start) * 1000
    print(f"\n   [首字 {first_ms or 0:.0f} ms | 总耗时 {total_ms:.0f} ms]\n")
    return "".join(parts)


async def main() -> None:
    if not API_KEY:
        sys.exit("没找到 LLM_API_KEY：本地写进 .env，网页里在 ⚙️ 设置 填 Key")
    client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=60)
    messages: list[dict] = [{"role": "system", "content": SYSTEM}]
    questions = [
        "我叫 Jiawei，是 Java 后端程序员。用一句话告诉我 Agent 是什么。",
        "和我平时写的 Spring 定时任务比，最大的区别是什么？",
        "我叫什么名字？我是做什么的？",           # 考验"记忆"
    ]
    try:
        for i, q in enumerate(questions, 1):
            messages.append({"role": "user", "content": q})
            print(f"第 {i} 轮（这次发出 {len(messages)} 条消息）\n你：{q}\n助手：", end="", flush=True)
            answer = await ask_stream(client, messages)
            messages.append({"role": "assistant", "content": answer})
    except AuthenticationError:
        sys.exit("401：API Key 不对或已失效")
    except APIConnectionError as e:
        sys.exit(f"连不上 {BASE_URL}：{e}")
    except APIStatusError as e:
        sys.exit(f"HTTP {e.status_code}：{e.message}")


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await